# Bölüm 2 — Modelleme: Parametre Adından Kategori Tahmini

**Proje:** GPT Eklenti Ekosisteminde Gizlilik Riski Analizi ve Otomatik Sınıflandırma

**Araştırma Sorusu 3:** Sadece parametre adına (ve varsa açıklamasına) bakarak, o parametrenin hangi kategoriye ait olduğu ne doğrulukla tahmin edilebilir?

**Hedef değişken kararı (Bölüm 1 sonunda verildi):** Ana model `main_data_type` (25, dengeli ve yeterli örneklemli sınıf) üzerine kurulacak — sağlam bir baseline, güvenilir metrikler ve yorumlanabilir feature importance için. Bölüm 2'nin sonunda, aynı pipeline'ı nadir sınıfları gruplayarak `data_type` (145 ince sınıf) üzerinde de deneyeceğiz; bu ikinci deneme Bölüm 3'teki "Other" sınıflandırması (Araştırma Sorusu 5) için de temel oluşturacak.

**Bu bölümün planı:**
1. Problemi kurma: metin özelliği (`name` + `description`) ve hedef (`main_data_type`) tanımlama, train/test ayırma
2. TF-IDF + Lojistik Regresyon (baseline model)
3. Embedding tabanlı model
4. İki modelin karşılaştırılması, feature importance yorumu
5. Aynı pipeline'ın `data_type` (nadir sınıflar gruplanmış) üzerinde denenmesi

## Adım 1 — Problemi Kurma: Girdi/Çıktı Tanımı ve Train/Test Ayrımı

Bir sınıflandırma modeli kurmadan önce iki şeyi netleştirmemiz lazım: **modele ne veriyoruz (X)** ve **modelden ne bekliyoruz (y)**.

- **X (girdi metni):** `name` ve `description`'ı tek bir metinde birleştireceğiz. Bunun sebebi basit: kayıtların %15.6'sında `description` boş (Bölüm 1'de gördük), o zaman modelin elinde sadece `name` kalıyor. İkisini birleştirip tek bir metin sütunu yapmak hem eldeki bilgiyi kaybetmeden kullanmamızı sağlıyor hem de pipeline'ı sadeleştiriyor.
- **y (hedef):** `main_data_type` — 25 sınıf.

Sonra veriyi **train/test** olarak ikiye ayıracağız — modelin daha önce hiç görmediği kayıtlar üzerinde ne kadar başarılı olduğunu ölçebilmek için. Sınıflar dengesiz olduğunu biliyoruz (en büyüğü 2568, en küçüğü 26 kayıt); bu yüzden **stratified split** kullanacağız, yani her sınıfın train ve test kümelerinde yaklaşık aynı oranda temsil edilmesini garanti edeceğiz. Aksi halde küçük bir sınıfın tüm örnekleri şans eseri train'e (ya da test'e) gidebilir ve o sınıf için hiç sağlıklı ölçüm yapamayız.

In [1]:
import json
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

DATA_PATH = '../backend/data_entries_final.json'

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
df['text'] = (df['name'].fillna('') + '. ' + df['description'].fillna('')).str.strip()

X = df['text']
y = df['main_data_type']

print('Toplam kayıt:', len(df))
print('Benzersiz sınıf sayısı:', y.nunique())
df[['name', 'description', 'text', 'main_data_type']].head(3)

Toplam kayıt: 12811
Benzersiz sınıf sayısı: 25


,name,description,text,main_data_type
0,version,The model version,version. The model version,App metadata
1,input,,input.,App usage data
2,prediction_id,,prediction_id.,Identifier


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train boyutu:', X_train.shape[0])
print('Test boyutu:', X_test.shape[0])
print()
print('En küçük sınıfın train/test dağılımı (Weather information):')
print('  train:', (y_train == 'Weather information').sum())
print('  test :', (y_test == 'Weather information').sum())

Train boyutu: 10248
Test boyutu: 2563

En küçük sınıfın train/test dağılımı (Weather information):
  train: 21
  test : 5


## Adım 2 — Baseline Model: TF-IDF + Lojistik Regresyon

**TF-IDF sezgisi:** Bir kelimenin bir kategoriyi ayırt etmede ne kadar işe yaradığını ölçüyoruz. Mantık şu: bir kelime (örn. "password") veri setinin genelinde nadir geçiyor ama geçtiği yerlerde hep aynı kategoriyle (örn. `Security credentials`) birlikte görünüyorsa, bu kelime güçlü bir sinyaldir. Buna karşılık "the", "data", "value" gibi her yerde geçen kelimeler ayırt edici değildir — TF-IDF bu tür kelimelere otomatik olarak düşük ağırlık verir. Sonuçta her metin, hangi kelimelerin ne kadar öne çıktığını gösteren bir sayı listesine (vektöre) dönüşür.

**Neden Lojistik Regresyon ile başlıyoruz:** Hızlı eğitilir, karmaşık değildir ve her kelimenin her kategoriye ne yönde katkı yaptığını (katsayılar üzerinden) doğrudan görebiliriz — bu da ileride "model neye bakarak karar veriyor" sorusunu cevaplarken işimize yarayacak. Asıl amacı karmaşık modellere geçmeden önce **makul bir referans noktası** oluşturmak: embedding tabanlı model bundan daha iyi çıkmazsa, ekstra karmaşıklığa değmez demektir.

TF-IDF'i **sadece train verisiyle** öğreteceğiz (`fit`), test verisine sadece uygulayacağız (`transform`) — aksi halde test setindeki bilgi sızıntısı (data leakage) yaşar, gerçekte olduğundan daha iyi bir sonuç görürüz.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print('Kelime dağarcığı boyutu:', len(tfidf.vocabulary_))
print('Train matris boyutu:', X_train_tfidf.shape)
print('Test matris boyutu:', X_test_tfidf.shape)

Kelime dağarcığı boyutu: 5000
Train matris boyutu: (10248, 5000)
Test matris boyutu: (2563, 5000)


In [4]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_tfidf, y_train)

y_pred_baseline = baseline_model.predict(X_test_tfidf)

**Neden hem accuracy hem macro F1'e bakıyoruz:** Sınıflar dengesiz olduğu için sadece accuracy'e bakmak yanıltıcı olabilir — model sadece en büyük sınıfları ("App usage data" gibi) doğru tahmin etse bile accuracy yüksek çıkabilir, küçük sınıflarda tamamen başarısız olsa bile. **Macro F1**, her sınıfı büyüklüğünden bağımsız eşit ağırlıklandırır, bu yüzden nadir sınıflardaki performansı da görünür kılar.

In [5]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

acc_baseline = accuracy_score(y_test, y_pred_baseline)
f1_macro_baseline = f1_score(y_test, y_pred_baseline, average='macro')
f1_weighted_baseline = f1_score(y_test, y_pred_baseline, average='weighted')

print(f'Accuracy: {acc_baseline:.3f}')
print(f'Macro F1: {f1_macro_baseline:.3f}')
print(f'Weighted F1: {f1_weighted_baseline:.3f}')
print()
print(classification_report(y_test, y_pred_baseline, zero_division=0))

Accuracy: 0.689
Macro F1: 0.468
Weighted F1: 0.692

                                precision    recall  f1-score   support

                  App metadata       1.00      0.36      0.53        28
                App usage data       0.70      0.81      0.75       514
               E-commerce data       0.00      0.00      0.00        13
             Event information       1.00      0.50      0.67        10
           Files and documents       0.67      0.43      0.53        83
           Finance information       1.00      0.14      0.25        28
Food and nutrition information       0.00      0.00      0.00         7
                   Gaming data       0.00      0.00      0.00         5
            Health information       1.00      0.44      0.61        16
                    Identifier       0.85      0.69      0.76       378
Legal and law enforcement data       0.00      0.00      0.00         5
                      Location       0.87      0.76      0.81       140
           

**İlk gözlemler (kesin karşılaştırma embedding modelinden sonra):**
- Accuracy %68.9, macro F1 sadece %46.8 — aradaki fark tam beklediğimiz gibi: model büyük sınıflarda iyi, küçük sınıflarda zayıf.
- Birçok küçük sınıfta precision 1.00 ama recall düşük (örn. `Travel information`: precision 1.00, recall 0.08) — yani model bu sınıfları tahmin ettiğinde haklı çıkıyor, ama çoğu zaman hiç tahmin etmiyor, daha büyük bir sınıfa kayıyor.
- `Other` sınıfı tam tersi bir örüntü gösteriyor: precision 0.36, recall 0.69 — yani model emin olamadığı birçok kaydı "Other" sınıfına atıyor, bu da onu bir tür "çöp kutusu" haline getiriyor. Bu, Bölüm 3'teki "Other" sınıflandırması için önemli bir uyarı işareti.
- Hassas kategorilerden `Security credentials` (F1 0.85) ve `Health information` (F1 0.61) şaşırtıcı derecede iyi ayırt edilebiliyor — muhtemelen "password", "api_key", "symptom" gibi kelimeler çok belirgin sinyaller.